# 03 — BERT finomhangolás (Google Colab, GPU)

**Ez a notebook Google Colabban fut, NEM a laptopodon.**

## Indulás előtt — 3 lépés

1. Töltsd fel ezt a notebookot: [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook**
2. Kapcsold be a GPU-t: **Runtime → Change runtime type → Hardware accelerator: T4 GPU** → Save
3. Töltsd fel a `data/processed/` mappából a három parquet fájlt (bal oldali mappa ikon → feltöltés),
   VAGY csatold a Google Drive-ot az alábbi cellával.

> **Miért nem a laptopon?** A BERT-base 110 millió paraméteres. 50 000 review
> 3 epochnyi finomhangolása CPU-n **8–15 óra**, egy ingyenes Colab T4 GPU-n
> **15–25 perc**. Ugyanaz a kód, 30× gyorsabban.

> **Colab időkorlátok:** az ingyenes verzió ~90 perc inaktivitás után, és
> legfeljebb ~12 óra futás után bont. Ezért mentünk checkpointot Drive-ra,
> és ezért nem futtatunk 3-nál több epochot.

## 1. Környezet ellenőrzése

In [ ]:
# Fut-e egyáltalán GPU-n? Ha itt "CPU"-t látsz, állítsd át:
# Runtime -> Change runtime type -> T4 GPU
import torch

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("!!! NINCS GPU — állítsd át a runtime típusát, különben órákig fog futni !!!")

In [ ]:
# Csak Colabon kell. Helyi futtatásnál hagyd ki.
!pip install -q transformers datasets accelerate evaluate
print("Telepítés kész.")

In [ ]:
import os, time, json
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset

# --- Reprodukálhatóság ---
SEED = 42
import random
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# --- Beállítások (a projekt config.py-jával összhangban) ---
MODEL_NAME    = "bert-base-uncased"
MAX_LENGTH    = 128
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3
LABELS        = ["negative", "neutral", "positive"]
LABEL2ID      = {l: i for i, l in enumerate(LABELS)}
ID2LABEL      = {i: l for l, i in LABEL2ID.items()}

print(f"Modell: {MODEL_NAME} | max_len {MAX_LENGTH} | batch {BATCH_SIZE} | lr {LEARNING_RATE}")

## 2. Adat betöltése

**A opció:** kézi feltöltés — húzd a három parquet fájlt a bal oldali fájlkezelőbe.
**B opció:** Google Drive csatolása (ajánlott, mert újraindítás után is megmarad).

In [ ]:
# --- B opció: Google Drive ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/szakdolgozat/data/processed"

# --- A opció: kézi feltöltés (alapértelmezett) ---
DATA_DIR = "/content"

train_df = pd.read_parquet(f"{DATA_DIR}/booking_train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}/booking_val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}/booking_test.parquet")

print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")
print(train_df["sentiment"].value_counts().to_string())

In [ ]:
# FONTOS: a 'text_bert' oszlopot használjuk, NEM a 'text_clean'-t!
#
# Miért? A BERT a kontextusból tanul. Ha eltávolítod a stopwordöket és a
# tagadást, a "not clean" -> "clean" lesz, és pont az ellenkezőjét tanítod.
# A text_bert csak a valódi zajt (HTML, URL) tisztította le.
#
# Ezt a döntést ÍRD LE a dolgozat 6.1 fejezetében — ez egy konkrét, indokolt
# módszertani különbség a két modellcsalád között.
TEXT_COL = "text_bert"

def to_hf_dataset(df):
    d = pd.DataFrame({
        "text": df[TEXT_COL].astype(str),
        "label": df["sentiment"].astype(str).map(LABEL2ID),
    })
    return Dataset.from_pandas(d, preserve_index=False)

ds_train = to_hf_dataset(train_df)
ds_val   = to_hf_dataset(val_df)
ds_test  = to_hf_dataset(test_df)
ds_train[0]

## 3. Tokenizálás

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    # padding itt NINCS: a DataCollator batchenként pad-el a leghosszabbig.
    # Így nem pazarolunk számítást a felesleges [PAD] tokenekre -> gyorsabb.
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds_train = ds_train.map(tokenize, batched=True, remove_columns=["text"])
ds_val   = ds_val.map(tokenize,   batched=True, remove_columns=["text"])
ds_test  = ds_test.map(tokenize,  batched=True, remove_columns=["text"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(ds_train)

In [ ]:
# Mennyi review lóg ki a 128 tokenes korláton? Ha sok, növeld a MAX_LENGTH-t
# (256-ra), de tudd: a futásidő ~négyzetesen nő a hosszal.
lens = [len(x) for x in ds_train["input_ids"]]
print(f"Token-hossz — átlag {np.mean(lens):.0f}, medián {np.median(lens):.0f}, "
      f"95%: {np.percentile(lens, 95):.0f}, max {max(lens)}")
print(f"A {MAX_LENGTH} tokenes korlátot a minták "
      f"{np.mean(np.array(lens) >= MAX_LENGTH):.1%}-a éri el (csonkolva lesz).")

## 4. Modell és metrikák

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)

n_total = sum(p.numel() for p in model.parameters())
print(f"Paraméterek száma: {n_total/1e6:.1f} millió")
print("(Összehasonlításul: a TF-IDF + LogReg modelled kb. 150 ezer súly.)")

In [ ]:
def compute_metrics(eval_pred):
    """UGYANAZOK a metrikák, mint a klasszikus modelleknél — enélkül
    az RQ1 összehasonlítás nem lenne korrekt."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0)
    _, _, f_wt, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": p_mac,
        "recall_macro": r_mac,
        "f1_macro": f_mac,
        "f1_weighted": f_wt,
    }

## 5. Tanítás

In [ ]:
args = TrainingArguments(
    output_dir="./bert_sentiment",
    seed=SEED,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,          # a tanulási ráta fokozatos felfuttatása

    # fp16: félpontosságú számítás. T4 GPU-n ~2x gyorsítás, alig mérhető
    # pontosságvesztéssel. Ha CPU-n futtatnád, ezt kapcsold ki.
    fp16=torch.cuda.is_available(),

    eval_strategy="epoch",     # régebbi transformers: evaluation_strategy
    save_strategy="epoch",
    logging_steps=100,

    # A legjobb modellt tartjuk meg, nem az utolsót — és a MACRO-F1 alapján,
    # mert a kiegyensúlyozatlan 3 osztályos feladatban ez a releváns metrika.
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,        # ne töltsük tele a Colab lemezt

    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print(f"Becsült lépésszám: {len(ds_train) // BATCH_SIZE * NUM_EPOCHS:,}")

In [ ]:
# ===== INNEN INDUL A TANÍTÁS =====
# 50 000 mintán, T4 GPU-n, 3 epoch: kb. 15-25 perc.
# NE zárd be a böngészőfület és ne hagyd magára 90+ percre!
t0 = time.time()
train_result = trainer.train()
train_minutes = (time.time() - t0) / 60

print(f"\nTanítási idő: {train_minutes:.1f} perc")
print("Ezt a számot JEGYEZD FEL — a dolgozat trade-off elemzéséhez kell "
      "(a klasszikus modellek másodpercek alatt tanultak).")

In [ ]:
# Tanulási görbe — jó ábra a dolgozatba, és megmutatja, van-e túltanulás.
import matplotlib.pyplot as plt

hist = pd.DataFrame(trainer.state.log_history)
tr = hist.dropna(subset=["loss"])
ev_ = hist.dropna(subset=["eval_loss"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tr["epoch"], tr["loss"], label="train loss", color="#4a7ba7")
axes[0].plot(ev_["epoch"], ev_["eval_loss"], "o-", label="val loss", color="#c1483b")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Tanulási görbe")
axes[0].legend(frameon=False)

axes[1].plot(ev_["epoch"], ev_["eval_f1_macro"], "o-", color="#3d7a5a", label="macro-F1")
axes[1].plot(ev_["epoch"], ev_["eval_accuracy"], "s--", color="#8a8680", label="accuracy")
axes[1].set(xlabel="Epoch", ylabel="Érték", title="Validációs teljesítmény")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig("03_bert_training_curve.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nHa a val loss emelkedni kezd, miközben a train loss csökken -> TÚLTANULÁS.")
print("Ilyenkor kevesebb epoch vagy nagyobb weight_decay kell. Írd le a dolgozatban!")

## 6. Kiértékelés a teszthalmazon

In [ ]:
test_out = trainer.predict(ds_test)
test_metrics = {k.replace("test_", ""): v for k, v in test_out.metrics.items()
                if k.startswith("test_") and isinstance(v, float)}

print("=== BERT — TESZT eredmények ===")
for k, v in test_metrics.items():
    print(f"  {k:22s} {v:.4f}")

y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = np.array(ds_test["label"])

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cmn = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(cmn, cmap="Blues")
ax.set_xticks(range(3), LABELS, rotation=20); ax.set_yticks(range(3), LABELS)
ax.set(xlabel="Predikált", ylabel="Valós", title="BERT — confusion matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cmn[i,j]:.1%}\n(n={cm[i,j]})", ha="center", va="center",
                fontsize=9, color="white" if cmn[i,j] > cmn.max()/2 else "black")
plt.tight_layout()
plt.savefig("03_bert_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. RQ1 — A nagy összehasonlítás

In [ ]:
# Írd ide a 02-es notebook eredményeit (results/tables/02_classical_validation_results.csv)
CLASSICAL = {
    "Multinomial NB":      {"accuracy": 0.000, "f1_macro": 0.000, "f1_weighted": 0.000},
    "Logistic Regression": {"accuracy": 0.000, "f1_macro": 0.000, "f1_weighted": 0.000},
    "Linear SVM":          {"accuracy": 0.000, "f1_macro": 0.000, "f1_weighted": 0.000},
}
CLASSICAL["BERT-base"] = {k: test_metrics.get(k, np.nan)
                          for k in ["accuracy", "f1_macro", "f1_weighted"]}

comp = pd.DataFrame(CLASSICAL).T.round(4)
comp["tanítási_idő_perc"] = [0.1, 0.5, 0.3, round(train_minutes, 1)]
display(comp)
comp.to_csv("03_rq1_model_comparison.csv")

print("\nEz a táblázat a dolgozat FŐ EREDMÉNYE (RQ1).")
print("Amit hozzá kell írni: nem csak AZT, melyik jobb, hanem MENNYIVEL és")
print("MILYEN ÁRON. Ha a BERT +5 pontot hoz 100x számításigényért, az egy")
print("valós mérnöki döntés, amit egy szállodalánc IT-vezetője is meghozna.")

## 8. Modell mentése

In [ ]:
SAVE_DIR = "./bert_sentiment_final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump({"test_metrics": test_metrics,
               "train_minutes": train_minutes,
               "config": {"model": MODEL_NAME, "max_length": MAX_LENGTH,
                          "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
                          "epochs": NUM_EPOCHS, "seed": SEED}}, f, indent=2)

# Tömörítés és letöltés (a Colab munkamenet megszűnésekor MINDEN TÖRLŐDIK!)
!zip -qr bert_sentiment_final.zip {SAVE_DIR}
from google.colab import files
files.download("bert_sentiment_final.zip")
files.download("03_rq1_model_comparison.csv")
files.download("03_bert_confusion_matrix.png")
files.download("03_bert_training_curve.png")

---

## Hibaelhárítás

| Hiba | Megoldás |
|---|---|
| `CUDA out of memory` | `BATCH_SIZE = 16` vagy `8`; vagy `MAX_LENGTH = 64` |
| `evaluation_strategy` ismeretlen | Régebbi transformers: `eval_strategy` → `evaluation_strategy` |
| Nagyon lassú (órák) | Nincs GPU. Runtime → Change runtime type → T4 GPU |
| Colab lecsatlakozik | Mentsd a checkpointokat Drive-ra (`output_dir` a Drive-on) |
| `f1_macro` nem nő 2. epoch után | Normális. 3 epoch a BERT-nél általában elég. |

## Amit érdemes még kipróbálni (ha van idő)

- **DistilBERT** (`distilbert-base-uncased`): 40%-kal kisebb, 60%-kal gyorsabb,
  a pontosság ~97%-át tartja. Egy külön sor a táblázatban → erős hatékonysági érv.
- **RoBERTa** (`roberta-base`): gyakran 1–2 ponttal jobb, mint a BERT.
- **Domain-specifikus modell**, pl. `nlptown/bert-base-multilingual-uncased-sentiment`.
- **Cross-dataset teszt:** futtasd a betanított BERT-et a TripAdvisor adaton (RQ5).

**Következő:** `04_absa_transformer.ipynb` — HRAST + sentence-pair ABSA (RQ3, RQ4).